In [3]:
import pybroker as pb
from pybroker import Strategy, StrategyConfig, indicator
import talib
import pandas as pd

# 1. 参数设置
STOCK_CODE = "600519.SH"  # 贵州茅台
INITIAL_CASH = 100000     # 初始资金10万元
SHORT_PERIOD = 5          # 短期均线周期
LONG_PERIOD = 20          # 长期均线周期
STOP_LOSS_PCT = 0.02      # 止损比例2%
TAKE_PROFIT_PCT = 0.05    # 止盈比例5%

# 2. 数据获取（使用AKShare数据源）
akshare = pb.ext.data.AKShare()
df = akshare.query(
    symbols=[STOCK_CODE],
    start_date="20200101",
    end_date="20231231",
    fields=["open", "high", "low", "close", "volume"]
)

# 3. 策略逻辑定义
def moving_average_strategy(ctx: pb.ExecContext):
    # 获取当前持仓状态
    position = ctx.long_pos()
    
    # 计算均线指标
    close_prices = ctx.data.close
    fast_ma = talib.SMA(close_prices, timeperiod=SHORT_PERIOD)
    slow_ma = talib.SMA(close_prices, timeperiod=LONG_PERIOD)
    
    # 生成交易信号
    if not position:  # 无持仓时
        if fast_ma[-1] > slow_ma[-1] and fast_ma[-2] <= slow_ma[-2]:  # 金叉买入
            ctx.buy_shares = ctx.calc_target_shares(0.3)  # 30%仓位
            ctx.stop_loss_pct = STOP_LOSS_PCT
            ctx.stop_profit_pct = TAKE_PROFIT_PCT
    else:  # 持仓时
        if fast_ma[-1] < slow_ma[-1] and fast_ma[-2] >= slow_ma[-2]:  # 死叉卖出
            ctx.sell_shares = position.shares

# 4. 回测配置
config = StrategyConfig(
    initial_cash=INITIAL_CASH,
    fee_amount=0.0005,  # 万五交易费率
    commission=0.001    # 千分之一佣金
)

# 5. 创建策略实例
data_source = pb.DataSource(df)
strategy = Strategy(
    data_source=data_source,
    start_date="20200101",
    end_date="20231231",
    config=config
)

# 6. 添加执行策略
strategy.add_execution(
    fn=moving_average_strategy,
    symbols=[STOCK_CODE],
    indicators={
        "fast_ma": indicator("fast_ma", lambda data: talib.SMA(data.close, timeperiod=SHORT_PERIOD)),
        "slow_ma": indicator("slow_ma", lambda data: talib.SMA(data.close, timeperiod=LONG_PERIOD))
    }
)

# 7. 运行回测
result = strategy.backtest(warmup=20)  # 前20天作为预热期

# 8. 结果分析
print("=== 回测关键指标 ===")
print(f"总收益率: {result.metrics['total_return']:.2f}%")
print(f"年化收益率: {result.metrics['annualized_return']:.2f}%")
print(f"最大回撤: {result.metrics['max_drawdown']:.2f}%")
print(f"夏普比率: {result.metrics['sharpe_ratio']:.2f}")
print(f"交易次数: {result.metrics['trade_count']}")

ImportError: cannot import name 'njit' from 'numba' (unknown location)

In [1]:
import pybroker as pb


ImportError: cannot import name 'njit' from 'numba' (unknown location)